# 1. Data Loading & Cleaning

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv("data/S&P500.csv")
df.head()

In [ ]:
df = df.iloc[2:].copy()
df.head()

In [ ]:
df.rename(columns={"Price":"Date"},inplace=True)
df["Date"] = pd.to_datetime(df["Date"])
df.set_index("Date",inplace=True)
df.sort_index(inplace=True)
df.head()

In [ ]:
df = df.apply(pd.to_numeric)

In [ ]:
df.info()

In [ ]:
df.isnull().sum()

# 2.Return Construction

In [ ]:
price = df["Close"]
price.head()

In [ ]:
import matplotlib.pyplot as plt

price.plot(title="S&P500 Daily Prices",figsize= (10,4))
plt.show()

In [ ]:
returns = np.log(price).diff().dropna()
returns.name = "log_returns"
returns.head()

In [ ]:
returns.plot(title="S&P500 Daily Return",figsize= (10,4))
plt.show()

In [ ]:
returns.describe()

# 3. AR /MA/ ARMA Modeling

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf,plot_pacf

fig, axes = plt.subplots(1,2,figsize = (12,6))

plot_acf(returns, lags=30,ax=axes[0])
axes[0].set_title("ACF of returns")

plot_pacf(returns, lags=30,ax=axes[1])
axes[1].set_title("PACF of returns")

plt.tight_layout()
plt.show()

In [ ]:
from statsmodels.tsa.stattools import adfuller

adf_stat, p_value, _, _, crit_vals, _ = adfuller(returns)

print("ADF Statistic:", adf_stat)
print("p-value:", p_value)
print("Critical Values:", crit_vals)

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

models = {
    "AR(1)": ARIMA(returns, order=(1,0,0)).fit(),
    "MA(1)": ARIMA(returns, order=(0,0,1)).fit(),
    "ARMA(1,1)": ARIMA(returns, order=(1,0,1)).fit()
}

for name, model in models.items():
    print(name, "AIC:", model.aic)

In [ ]:
from statsmodels.stats.diagnostic import acorr_ljungbox

residuals = models["ARMA(1,1)"].resid

lb_test = acorr_ljungbox(residuals, lags=[10], return_df=True)
print(lb_test)

In [ ]:
from statsmodels.stats.diagnostic import het_arch

arch_test = het_arch(residuals, nlags=10)
print(f"ARCH LM Stat: {arch_test[0]}")
print(f"p-value: {arch_test[1]}")

## Time Series Diagnostics and Model Selection

The returns exhibit little linear correlation and are approxlimately white noise in the mean

Given the lack of strong autocorrelation structure, a simple AR(1),MA(1),ARMA(1,1) or constant mean specification is sufficient for modeling the conditional mean

First we conducted ADF test where we got p-value << 0.05 so we reject the null hypothesis of non-stationarity (unit root)

Then on the basis of AIC value we selected ARMA(1,1) model and then performed Ljung-Box test on the residuals which tells us that there is still autocorrelation in the residuals so we did an ARCH test to find out whether we should use the GARCH or ARMA-GARCH model or not as the p-value of ARCH test was less than << 0.5 we concluded that GARCH type model was needed as GARCH handle variance

# 4.AR-GRACH modeling

In [ ]:
import sys
!{sys.executable} -m pip install arch

In [ ]:
from arch import arch_model

model = arch_model(
    returns * 100,
    mean='ARX',
    lags=1,
    vol='GARCH',
    p=1,
    q=1,
    dist='t'
)

res = model.fit(update_freq=5, disp='off')
print(res.summary())

In [ ]:
std_resid = res.std_resid.dropna()

lb = acorr_ljungbox(std_resid, lags=[10], return_df=True)
print(lb)

In [ ]:
arch_test = het_arch(std_resid, nlags=10)
print("ARCH LM:", arch_test[0])
print("p-value:", arch_test[1])

In [ ]:
plot_acf(std_resid**2, lags=30)
plt.title("ACF of Squared Standardized Residuals")
plt.show()

In [ ]:
import seaborn as sns

sns.histplot(std_resid,kde=True,stat="density")
plt.title("Histplot with kde of std. residue")
plt.show()

## Model Specification and Diagnostic Tests

We use AR(1)-GARCH(1,1) model on the return to make in not to complex but enough to do justice to the data

Then we again ran the Ljung test on the standard residue which we could conclude that the residue does not have any autocorrelation. the test result indicate that there is no more autocorrelation in return

Then we did ARCH test again to check if there is any additional conditional heteroskedasticity remains. The test fails to reject the Null hypothesis indicating that the selected model sufficently explain the time varrying in the return

Then ACF confirms that the standard residue is approximately white noise within the 95% confidence bands 

# 5.Value at Risk(VaR) and Backtesting

In [ ]:
forecast = res.forecast(horizon=10)
forecast.variance.iloc[-1]

In [ ]:
vol_forecast = np.sqrt(forecast.variance.iloc[-1])
vol_forecast

In [ ]:
annual_vol = vol_forecast * np.sqrt(252)
annual_vol

In [ ]:
from scipy.stats import norm

z_95 = norm.ppf(0.05)
VaR_95 = z_95 * vol_forecast.iloc[0]
VaR_95

In [ ]:
from scipy.stats import t as student_t
import numpy as np

violations = []
VaRs = []
actuals = []

window = 1000  # e.g. ~4 years of daily data

returns_scaled = returns * 100

for i in range(window, len(returns_scaled) - 1):
    train = returns_scaled.iloc[:i]
    
    model = arch_model(
        train,
        mean="AR",
        lags=1,
        vol="GARCH",
        p=1,
        q=1,
        dist="t",
        rescale= False
    )
    res = model.fit(disp="off")
    
    forecast = res.forecast(horizon=1)
    vol = np.sqrt(forecast.variance.iloc[-1, 0])
    
    nu = res.params["nu"]
    z = student_t.ppf(0.05, nu)
    
    VaR = z * vol
    actual = returns_scaled.iloc[i + 1]
    
    VaRs.append(VaR)
    actuals.append(actual)
    violations.append(actual < VaR)

In [ ]:
from scipy.stats import chi2

violations = np.array(violations).astype(int)
x = violations.sum()
T = len(violations)
alpha = 0.05

LR_uc = -2 * (
    (T - x) * np.log((1 - alpha) / (1 - x/T)) +
    x * np.log(alpha / (x/T))
)

p_value = 1 - chi2.cdf(LR_uc, df=1)

print("Violations:", x)
print("Expected:", alpha * T)
print("LR statistic:", LR_uc)
print("p-value:", p_value)

In [ ]:
N01 = N11 = N10 = N00 = 0

for i in range(1,len(violations)):
    if violations[i-1] == 0 and violations[i] == 0:
        N00 += 1
    elif violations[i-1] == 1 and violations[i] == 0:
        N01 += 1
    elif violations[i-1] == 0 and violations[i] == 1:
        N10 += 1
    elif violations[i-1] == 1 and violations[i] == 1:
        N11 += 1


pi_01 = N01/(N00+N01) if (N00+N01) > 0 else 0
pi_11 = N11/(N10+N11) if (N10+N11) > 0 else 0
pi = (N01+N11)/(N00+N01+N10+N11)

In [ ]:
LR_ind = -2 * (
    (N00 + N01) * np.log(1 - pi) +
    (N10 + N11) * np.log(1 - pi) +
    (N01 + N11) * np.log(pi)
    -
    (
        N00 * np.log(1 - pi_01) +
        N01 * np.log(pi_01) +
        N10 * np.log(1 - pi_11) +
        N11 * np.log(pi_11)
    )
)

p_value_ind = 1 - chi2.cdf(LR_ind, df=1)

print("Christoffersen LR_ind:", LR_ind)
print("p-value:", p_value_ind)

## Value at Risk (VaR) Estimation and Backtesting

We estimated the one-day Value at Risk (VaR) for the model as:
**VaR = -1.006888091562848**
This implies that, at the chosen confidence level, the maximum expected daily loss is approximately **1.01**

Then we did VaR backtesting with the Kupiec test which tells us that we are overestimating the VaR making it overly conservative are the actual number of violationare less than the expected number of violation

Then we did Chirstoffersen Test to check that do violation occur in clusters. This test told us that the violation do occur in clusters which indicates that the model fails to capture the time-varrying risk dynamics

# 6.Visualization

In [ ]:
plt.figure(figsize=(14,5))
plt.plot(actuals, label="Returns", alpha=0.6)
plt.plot(VaRs, label="VaR (5%)", color="red")
plt.scatter(
    [i for i,v in enumerate(violations) if v],
    [actuals[i] for i,v in enumerate(violations) if v],
    color="black", s=20, label="Violations"
)

plt.legend()
plt.title("Returns vs 1-Day 95% VaR")
plt.show()


In [ ]:
plt.figure(figsize=(14,5))
plt.plot(violations, drawstyle="steps-post")
plt.title("VaR Violations Over Time")
plt.yticks([0,1])
plt.show()

In [ ]:
import pandas as pd

rolling_rate = pd.Series(violations).rolling(250).mean()

plt.figure(figsize=(14,3))
plt.plot(rolling_rate)
plt.axhline(0.05, color="red", linestyle="--", label="Expected 5%")
plt.title("Rolling Violation Rate (1 Year)")
plt.legend()
plt.show()

# Conclusion & Risk Interpretation

This project demonstrates a full VaR modeling pipeline using AR–GARCH models.
While unconditional coverage is conservative (Kupiec test),
the Christoffersen test reveals violation clustering,
indicating limitations of the conditional volatility dynamics.
This highlights the importance of independence testing in VaR backtesting.
